In [1]:
import pandas as pd
import glob
import numpy as np

In [2]:
account = pd.read_csv('accounts.csv')
products = pd.read_csv('products.csv')
sales_pipeline = pd.read_csv('sales_pipeline.csv')
sales_teams = pd.read_csv('sales_teams.csv')

In [ ]:
file = pd.merge(sales_pipeline, account, on='account', how='left').merge(products, on='product', how='left').merge(sales_teams, on='sales_agent', how='left')
file = pd.DataFrame(file)
file.head()

file.to_csv('hubspot_merge_file.csv')

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,sector,year_established,revenue,employees,office_location,subsidiary_of,series,sales_price,manager,regional_office
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,retail,2001.0,718.62,2448.0,United States,NaN,GTX,1096.0,Dustin Brinkmann,Central
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0,medical,2002.0,3178.24,4540.0,United States,NaN,NaN,NaN,Melvin Marxen,Central
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,retail,2001.0,718.62,2448.0,United States,NaN,MG,55.0,Melvin Marxen,Central
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,software,1998.0,2714.90,2641.0,United States,Acme Corporation,GTX,550.0,Dustin Brinkmann,Central
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,services,1982.0,792.46,1299.0,United States,NaN,GTX,550.0,Summer Sewald,West


In [17]:
file.info()

<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   opportunity_id    8800 non-null   str    
 1   sales_agent       8800 non-null   str    
 2   product           8800 non-null   str    
 3   account           7375 non-null   str    
 4   deal_stage        8800 non-null   str    
 5   engage_date       8300 non-null   str    
 6   close_date        6711 non-null   str    
 7   close_value       6711 non-null   float64
 8   sector            7375 non-null   str    
 9   year_established  7375 non-null   float64
 10  revenue           7375 non-null   float64
 11  employees         7375 non-null   float64
 12  office_location   7375 non-null   str    
 13  subsidiary_of     1292 non-null   str    
 14  series            7320 non-null   str    
 15  sales_price       7320 non-null   float64
 16  manager           8800 non-null   str    
 17  region

In [23]:
file.describe()

,close_value,year_established,revenue,employees,sales_price
count,6711.000000,7375.000000,7375.000000,7375.000000,7320.000000
mean,1490.915512,1995.483661,2467.515536,5701.213424,1885.394126
std,2320.670773,9.187126,2596.135671,6816.683924,2619.399523
min,0.000000,1979.000000,4.540000,9.000000,55.000000
25%,0.000000,1988.000000,647.180000,1238.000000,550.000000
50%,472.000000,1995.000000,1698.200000,3492.000000,1096.000000
75%,3225.000000,2002.000000,2952.730000,7523.000000,3393.000000
max,30288.000000,2017.000000,11698.030000,34288.000000,26768.000000


In [34]:
file.select_dtypes('str').describe()

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,sector,office_location,subsidiary_of,series,manager,regional_office
count,8800,8800,8800,7375,8800,8300,6711,7375,7375,1292,7320,8800,8800
unique,8800,30,7,85,4,421,306,10,15,7,3,6,3
top,1C1I7A6R,Darcel Schlecht,GTX Basic,Hottechi,Won,2017-07-22,2017-05-22,retail,United States,Acme Corporation,GTX,Melvin Marxen,Central
freq,1,747,1866,200,4238,66,41,1397,6120,322,4217,1929,3512


In [136]:
def data_cleaning(df):
    # read the csv file
    df = pd.read_csv(df)

    # drop the subsidiary_of column
    drop_col = df.drop(columns = ['subsidiary_of'], inplace = True)

    # rename the columns
    df = df.rename(columns = {'account':'company_name'.capitalize(),
                              'revenue':'revenue ($M)'.capitalize(),
                              'employees':'employees'.capitalize(),
                              'office_location':'office'.capitalize(),
                              'product':'product_name'.capitalize(),
                              'series':'product_series'.capitalize(),
                              'sales_price':'sales_price ($)'.capitalize(),
                              'opportunity_id':'id'.upper(),
                              'close_value':'close_value ($)'.capitalize(),
                              'close_date':'close_date'.capitalize(),
                              'engage_date':'engage_date'.capitalize(),
                              'sales_agent':'sales_agent'.capitalize(), 
                              'deal_stage':'deal_stage'.capitalize(),})
    
    # converting data types
    
    df['Close_date'] = pd.to_datetime(df['Close_date'], errors='coerce')
    df['Revenue ($m)'] = pd.to_numeric(df['Revenue ($m)'])
    df['Sales_price ($)'] = pd.to_numeric(df['Sales_price ($)'], errors='coerce')
    df['Engage_date'] = pd.to_datetime(df['Engage_date'], errors='coerce')
    df['Close_value ($)'] = pd.to_numeric(df['Close_value ($)'], errors='coerce')

    # Drop nan values in the 'Company_name' column
    df = df.dropna(subset = ['Company_name'])
    
    # drop the index column
    df = df.drop(columns = ['Unnamed: 0'])
    return df

In [137]:
file = data_cleaning('hubspot_merge_file.csv')
file.head()

,ID,Sales_agent,Product_name,Company_name,Deal_stage,Engage_date,Close_date,Close_value ($),sector,year_established,Revenue ($m),Employees,Office,Product_series,Sales_price ($),manager,regional_office
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,retail,2001.0,718.62,2448.0,United States,GTX,1096.0,Dustin Brinkmann,Central
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0,medical,2002.0,3178.24,4540.0,United States,NaN,NaN,Melvin Marxen,Central
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,retail,2001.0,718.62,2448.0,United States,MG,55.0,Melvin Marxen,Central
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,software,1998.0,2714.90,2641.0,United States,GTX,550.0,Dustin Brinkmann,Central
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,services,1982.0,792.46,1299.0,United States,GTX,550.0,Summer Sewald,West


In [138]:
file.info()

<class 'pandas.DataFrame'>
Index: 7375 entries, 0 to 8791
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ID                7375 non-null   str           
 1   Sales_agent       7375 non-null   str           
 2   Product_name      7375 non-null   str           
 3   Company_name      7375 non-null   str           
 4   Deal_stage        7375 non-null   str           
 5   Engage_date       7212 non-null   datetime64[us]
 6   Close_date        6711 non-null   datetime64[us]
 7   Close_value ($)   6711 non-null   float64       
 8   sector            7375 non-null   str           
 9   year_established  7375 non-null   float64       
 10  Revenue ($m)      7375 non-null   float64       
 11  Employees         7375 non-null   float64       
 12  Office            7375 non-null   str           
 13  Product_series    6117 non-null   str           
 14  Sales_price ($)   6117 non-null   float6

In [176]:
file.groupby('Sales_price ($)').agg({'ID':'count', 'Product_series':'unique', 'Product_name':'unique'}).sort_values('Sales_price ($)', ascending = False)

,ID,Product_series,Product_name
Sales_price ($),,,
26768.0,32,[GTK],[GTK 500]
5482.0,814,[GTX],[GTX Plus Pro]
3393.0,1181,[MG],[MG Advanced]
1096.0,1168,[GTX],[GTX Plus Basic]
550.0,1563,[GTX],[GTX Basic]
55.0,1359,[MG],[MG Special]


In [166]:
file.groupby('Product_name')['Sales_price ($)'].unique()

Product_name
GTK 500           [26768.0]
GTX Basic           [550.0]
GTX Plus Basic     [1096.0]
GTX Plus Pro       [5482.0]
GTXPro                [nan]
MG Advanced        [3393.0]
MG Special           [55.0]
Name: Sales_price ($), dtype: object

In [169]:
file.groupby(file['Product_name'] == 'GTXPro').agg({'Sales_price ($)':'count'})

,Sales_price ($)
Product_name,
False,6117
True,0


In [173]:
file[file['Product_name'] == 'GTXPro'].isnull().sum()#.head()#.groupby('Sales_price ($)').agg({'ID':'count', 'Product_name':'unique'}).sort_values('Sales_price ($)', ascending = False)

ID                     0
Sales_agent            0
Product_name           0
Company_name           0
Deal_stage             0
Engage_date           34
Close_date           111
Close_value ($)      111
sector                 0
year_established       0
Revenue ($m)           0
Employees              0
Office                 0
Product_series      1258
Sales_price ($)     1258
manager                0
regional_office        0
dtype: int64